In [15]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

## PIPELINE

El Pipeline convierte age y gender en variables binarias (OneHot), normaliza todas las variables numéricas (StandardScaler) y entrena un modelo de clasificación automáticamente. 

In [16]:
X = pd.read_csv("../data/clean/X_clean.csv")
y = pd.read_csv("../data/clean/y.csv").squeeze()

añadimos categoricas

In [17]:

cat_cols = ["age", "gender", "merchant", "customer", "category"]
num_cols = X.drop(columns=cat_cols).columns.tolist()

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42
    )
}

In [21]:
import time 

def train_base_models(models):
    results = {}

    for name, model in models.items():
        
        clf = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("model", model)
        ])
        
        # inicio
        start = time.perf_counter()

        clf.fit(X_train, y_train)

        # fin
        end = time.perf_counter()
        train_time = end - start

        # Predicción
        start_pred = time.perf_counter()

        y_pred_proba = clf.predict_proba(X_test)[:, 1]

        end_pred = time.perf_counter()
        pred_time = end_pred - start_pred
        
        # metrica
        auc = roc_auc_score(y_test, y_pred_proba)
        results[name] = {
            "AUC": auc,
            "Train Time (s)": train_time,
            "Predict Time (s)": pred_time
        }
        
        print(f"{name} → AUC: {auc:.4f} | Train: {train_time:.2f}s | Predict: {pred_time:.4f}s")

In [22]:
train_base_models(models)

RandomForest → AUC: 0.9952 | Train: 52.68s | Predict: 0.2180s
XGBoost → AUC: 0.9975 | Train: 6.62s | Predict: 0.0908s
[LightGBM] [Info] Number of positive: 469954, number of negative: 469954
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.308949 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 72090
[LightGBM] [Info] Number of data points in the train set: 939908, number of used features: 4174
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/opt/miniconda3/envs/tfg_env/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM → AUC: 0.9980 | Train: 7.48s | Predict: 0.3229s


In [23]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="AUC", ascending=False)

print(results_df)

                   AUC  Train Time (s)  Predict Time (s)
LightGBM      0.997961        7.389346          0.333923
XGBoost       0.997510        6.679053          0.098710
RandomForest  0.995239       52.279186          0.219570


## AJUSTAR HIPERPARÁMETROS